# 💬 Chat With Your Fine-Tuned Model

Load your trained model from Google Drive and chat with it directly here.

**Requirements:**
- T4 GPU (Runtime → Change runtime type → T4 GPU)
- Training notebook must have completed and saved to Google Drive

**Time:** ~3 minutes to load, then instant responses

## ⚙️ Configuration

**Only change MODEL_NAME — must match exactly what you used in training.**

In [ ]:
# ============================================================================
# CONFIGURATION — ONLY UPDATE MODEL_NAME
# ============================================================================

MODEL_NAME = "finance-ai-v1"  # <- CHANGE THIS to match your training notebook

# Chat settings
TEMPERATURE = 0.7
MAX_TOKENS  = 256

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

DRIVE_ROOT       = "/content/drive/MyDrive/Finetune_Jobs"
MODEL_OUTPUT_DIR = f"{DRIVE_ROOT}/models/{MODEL_NAME}"

print("✅ Configuration loaded")
print(f"   Model: {MODEL_NAME}")
print(f"   Path : {MODEL_OUTPUT_DIR}")


## 🔗 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')

if not os.path.exists(MODEL_OUTPUT_DIR):
    raise FileNotFoundError(
        f"
❌ Model not found: {MODEL_OUTPUT_DIR}
"
        f"Check MODEL_NAME matches your training notebook exactly.
"
        f"Available models:
"
        + "
".join([f"  - {m}" for m in os.listdir(f"{DRIVE_ROOT}/models")])
        if os.path.exists(f"{DRIVE_ROOT}/models") else ""
    )

# Load system prompt from training config
config_path = f"{MODEL_OUTPUT_DIR}/training_config.json"
with open(config_path) as f:
    config = json.load(f)
SYSTEM_PROMPT = config.get("system_prompt", "").strip()

print(f"✅ Drive mounted")
print(f"✅ Model folder found")
print(f"✅ System prompt loaded:")
print(f"   {SYSTEM_PROMPT[:100]}...")

print(f"
📁 Model files:")
for f in sorted(os.listdir(MODEL_OUTPUT_DIR)):
    size = os.path.getsize(f"{MODEL_OUTPUT_DIR}/{f}") / (1024**2)
    print(f"   {f:<45} ({size:>8.1f} MB)")


## 📦 Step 2: Install Unsloth

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
print("✅ Unsloth installed")


## 🤖 Step 3: Load Your Trained Model

**Takes 2-3 minutes. Please wait.**

In [ ]:
from unsloth import FastLanguageModel
import torch

print("⏳ Loading your trained model from Google Drive...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_OUTPUT_DIR,
    max_seq_length = 1024,
    dtype          = None,
    load_in_4bit   = True,
)

FastLanguageModel.for_inference(model)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded successfully!")
print(f"   Model : {MODEL_NAME}")
print(f"   Ready for chat")


## 💬 Step 4: Chat Function

In [ ]:
def chat(message, history=None):
    """
    Send a message and get a response from your trained model.
    history: list of {role, content} dicts for multi-turn conversation
    """
    if history is None:
        history = []

    # Build prompt manually — avoids Unsloth shape errors
    prompt = f"<|start_header_id|>system<|end_header_id|>

{SYSTEM_PROMPT}<|eot_id|>"

    # Add conversation history
    for turn in history:
        role    = turn["role"]
        content = turn["content"]
        if role == "user":
            prompt += f"<|start_header_id|>user<|end_header_id|>

{content}<|eot_id|>"
        elif role == "assistant":
            prompt += f"<|start_header_id|>assistant<|end_header_id|>

{content}<|eot_id|>"

    # Add current message
    prompt += f"<|start_header_id|>user<|end_header_id|>

{message}<|eot_id|>"
    prompt += f"<|start_header_id|>assistant<|end_header_id|>

"

    # Tokenize
    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    input_len = inputs["input_ids"].shape[1]

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs["input_ids"],
            attention_mask     = inputs["attention_mask"],
            max_new_tokens     = MAX_TOKENS,
            temperature        = TEMPERATURE,
            top_p              = 0.9,
            do_sample          = True,
            repetition_penalty = 1.15,
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )

    # Decode only new tokens
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


print("✅ Chat function ready")
print("Usage: response = chat("your question here")")


## 🧪 Step 5: Quick Test

In [ ]:
import time

print("🤖 Testing your model...
")

test_q = "What is DCF valuation and when should you use it?"
print(f"👤 User: {test_q}")
print("⏳ Generating...
")

t0  = time.time()
ans = chat(test_q)
print(f"🤖 Bot: {ans}")
print(f"
⏱️  {time.time()-t0:.1f}s")
print("
✅ Model is working! Proceed to Step 6 for interactive chat.")


## 💬 Step 6: Interactive Chat

**Run this cell and start chatting!**
Type your message and press Enter. Type  to stop. Type  to clear history.

In [ ]:
import time

print("=" * 65)
print(f"💬 Chat with {MODEL_NAME}")
print("=" * 65)
print("Commands: quit = stop | reset = clear history
")

history      = []
MAX_HISTORY  = 5  # Keep last 5 turns to avoid GPU OOM

while True:
    user_input = input("
👤 You: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("
👋 Goodbye!")
        break
    if user_input.lower() == "reset":
        history = []
        print("   🔄 Conversation cleared.")
        continue

    t0       = time.time()
    response = chat(user_input, history=history)
    elapsed  = time.time() - t0

    print(f"
🤖 {MODEL_NAME}: {response}")
    print(f"   ⏱️  {elapsed:.1f}s")

    # Update history
    history.append({"role": "user",      "content": user_input})
    history.append({"role": "assistant", "content": response})

    # Trim to avoid context overflow
    if len(history) > MAX_HISTORY * 2:
        history = history[-(MAX_HISTORY * 2):]


## 📊 Step 7: Batch Test (Optional)

Test multiple questions at once.

In [ ]:
test_questions = [
    "What is the Sharpe ratio and how do you interpret it?",
    "Calculate DCF for 0M revenue, 20% growth, 5 years, 12% discount rate, 3% terminal growth",
    "What is the difference between ROE and ROIC?",
    "Company has P/E of 25, growth rate 15%. Is it overvalued?",
]

print("🧪 Batch Test
" + "="*65)

for i, q in enumerate(test_questions, 1):
    t0 = time.time()
    a  = chat(q)
    print(f"
{i}. 👤 {q}")
    print(f"   🤖 {a}")
    print(f"   ⏱️  {time.time()-t0:.1f}s")
    print("-"*65)
